# Student Health Risk — Playground Series S6E7

**Task**: classificazione multi-classe (`fit` / `at-risk` / `unhealthy`)
**Metrica**: **balanced accuracy** → ogni classe pesa uguale, indipendentemente dal supporto
**Dataset**: 690.088 righe, 7 feature numeriche + 6 categoriche, null su tutte le feature

## Log delle versioni
| ver | descrizione | OOF BA (argmax) | OOF BA (prior-corr, t*) | LB |
|-----|-------------|-----------------|--------------------------|-----|
| v1  | LightGBM baseline, NaN crudi, prior-correction | 0.87626 | **0.94938** (t=1.00) | **0.94591** ✓ |
| v2  | v1 + `bmi_is_missing` (ablation) | = v1 | **delta 0.00000** → scartata | — |
| v3  | Optuna (obiettivo: BA post prior-correction) | 0.87824 | **0.94979** (t=1.00) | **0.94961** ✓ |
| v4  | CatBoost (GPU), stessi fold | — | *(in corso)* | — |
| v5  | Blend LightGBM + CatBoost (pesi + t su OOF) | — | *(in corso)* | — |

## Decisioni chiave (e perché)
1. **Nessuna imputation nella baseline**: LightGBM gestisce i NaN nativamente (impara la direzione dello split); le categoriche trattano il NaN come categoria propria. Qualsiasi imputation dovrà *battere* questa baseline in ablation.
2. **Prior-correction**: con sbilanciamento 86/8/6 e balanced accuracy, l'argmax del posteriore è subottimale. Si usa `argmax P(c|x) / prior^t`. Sweep su t: **t*=1.00** → probabilità ben calibrate (stesso risultato di S6E6).
3. **Diagnostica missingness** (vedi sotto): missingness ≈ MCAR ovunque tranne `bmi`, il cui tasso di null varia 4x tra `unhealthy` (0.71%) e `fit` (2.86%) → unico indicatore candidato come feature (v2).
4. **Co-missingness `exercise_duration`/`diet_type`**: entrambe 6.901 null, ma overlap osservato 72 ≈ 69 atteso sotto indipendenza → stessi conteggi marginali, nessun pattern strutturato.

**Ancoraggio OOF↔LB verificato** (v1): gap 0.0035, fisiologico (public LB = frazione del test; t* scelto su OOF → lieve ottimismo). OOF su 690k righe = stima più affidabile del public LB.

**Nota v3**: delta OOF +0.00040 (LightGBM spremuto: da qui in poi il margine si chiama diversita', non tuning). t*=1.00 per la terza volta consecutiva → probabilita' LightGBM ben calibrate, correzione bayesiana pura ottimale. Il LB (+0.0037) sovrastima il delta reale: rumore favorevole del public split, la bussola resta l'OOF.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

import kagglehub
kagglehub.competition_download('playground-series-s6e7')
BASE = '/kaggle/input/competitions/playground-series-s6e7'

SEED = 42
N_FOLDS = 5
print('Setup ✓')

In [ ]:
train = pd.read_csv(f'{BASE}/train.csv')
test  = pd.read_csv(f'{BASE}/test.csv')
print(train.shape, test.shape)

## 1. EDA essenziale

Target fortemente sbilanciato: `at-risk` 85.9%, `unhealthy` 8.4%, `fit` 5.8%.
Con la balanced accuracy, il campo di battaglia sono le due classi minoritarie.

In [ ]:
display(train['health_condition'].value_counts())
display(train['health_condition'].value_counts(normalize=True).round(4))
train.isnull().sum()

## 2. Diagnostica della missingness

Tre domande, in ordine:
1. i null di colonne con conteggi identici cadono sulle stesse righe? (co-missingness)
2. la missingness è correlata tra colonne? (pattern strutturati)
3. la missingness correla col **target**? (→ feature gratis)

**Esito** (prima esecuzione): missingness ≈ MCAR ovunque, unica eccezione `bmi`
(null: 0.71% in `unhealthy`, 2.08% in `at-risk`, 2.86% in `fit`).

In [ ]:
# 1. co-missingness: exercise_duration e diet_type hanno entrambe 6901 null
overlap = (train['exercise_duration'].isnull() & train['diet_type'].isnull()).sum()
expected = train['exercise_duration'].isnull().sum() * train['diet_type'].isnull().sum() / len(train)
print(f'overlap osservato: {overlap} | atteso sotto indipendenza: {expected:.0f}')

# 2. correlazione della missingness tra colonne
msno = train.drop(columns=['id', 'health_condition']).isnull()
plt.figure(figsize=(9, 7))
sns.heatmap(msno.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlazione della missingness')
plt.show()

# 3. frazione di null per classe (la domanda da un milione di punti)
null_by_class = train.groupby('health_condition')[msno.columns].apply(lambda g: g.isnull().mean()).T
display(null_by_class.round(4))

## 3. Preprocessing minimale

- NaN numerici: lasciati crudi (LightGBM li gestisce nativamente)
- Categoriche: cast a `category` (il NaN diventa di fatto una categoria)
- Target: LabelEncoder → **usare sempre `le.classes_` per rimappare le predizioni**, mai a mano

In [ ]:
TARGET = 'health_condition'
cat_cols = train.select_dtypes('object').columns.drop(TARGET).tolist()

le = LabelEncoder()
y = le.fit_transform(train[TARGET])
print('ordine classi:', dict(enumerate(le.classes_)))

def make_X(df, add_bmi_missing=False):
    X = df.drop(columns=['id', TARGET], errors='ignore').copy()
    if add_bmi_missing:
        X['bmi_is_missing'] = X['bmi'].isnull().astype(int)
    for c in cat_cols:
        X[c] = X[c].astype('category')
    return X

X      = make_X(train)
X_test = make_X(test)
priors = np.bincount(y) / len(y)
print('priors:', priors.round(4))

## 4. Pipeline CV riusabile

Un'unica funzione per baseline e ablation: stessi fold, stesso seed → confronti puliti.
Restituisce OOF, modelli di fold e BA per fold.

In [ ]:
LGB_PARAMS = dict(
    objective='multiclass', num_class=3,
    n_estimators=2000, learning_rate=0.05,
    num_leaves=63, random_state=SEED,
    verbosity=-1,
)

def run_cv(X, y, params=LGB_PARAMS, n_folds=N_FOLDS, seed=SEED, label=''):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros((len(X), 3))
    models, fold_scores = [], []
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**params)
        model.fit(X.iloc[tr], y[tr],
                  eval_set=[(X.iloc[va], y[va])],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = model.predict_proba(X.iloc[va])
        models.append(model)
        ba = balanced_accuracy_score(y[va], oof[va].argmax(1))
        fold_scores.append(ba)
        print(f'[{label}] fold {fold}: BA = {ba:.5f}')
    print(f'[{label}] OOF BA (argmax): {balanced_accuracy_score(y, oof.argmax(1)):.5f}')
    return oof, models, fold_scores

def prior_sweep(oof, y, priors, ts=np.arange(0.0, 1.55, 0.25)):
    results = {}
    for t in ts:
        results[round(float(t), 2)] = balanced_accuracy_score(y, (oof / priors**t).argmax(1))
    for t, ba in results.items():
        print(f't={t:.2f}: BA = {ba:.5f}')
    t_star = max(results, key=results.get)
    print(f'--> t* = {t_star:.2f}, BA = {results[t_star]:.5f}')
    return t_star, results

## 5. v1 — baseline

Risultati prima esecuzione: fold BA 0.874–0.879 (CV stabilissima), OOF argmax **0.87626**,
prior-correction **+7.3 punti** → **0.94938** a t*=1.00.

In [ ]:
oof_v1, models_v1, _ = run_cv(X, y, label='v1')
t_star, sweep_v1 = prior_sweep(oof_v1, y, priors)

### Sweep fine intorno a t* (opzionale, per i perfezionisti)

In [ ]:
_ = prior_sweep(oof_v1, y, priors, ts=np.arange(0.85, 1.16, 0.05))

## 6. Submission (v1 + prior-correction)

Media delle `predict_proba` dei 5 modelli di fold, correzione con i prior del **train**, argmax,
rimappatura con `le.classes_`.

⚠️ Prima submission = **ancoraggio OOF ↔ LB**: se il LB ≈ 0.949, la macchina è verificata.

In [ ]:
T_SUB = 1.0  # aggiornare se lo sweep fine trova di meglio

test_proba = np.mean([m.predict_proba(X_test) for m in models_v1], axis=0)
test_pred  = le.classes_[(test_proba / priors**T_SUB).argmax(1)]

sub = pd.DataFrame({'id': test['id'], TARGET: test_pred})
sub.to_csv('submission.csv', index=False)
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()

## 7. v2 — ablation: `bmi_is_missing`

Ipotesi dalla diagnostica: il tasso di null di `bmi` varia 4x tra `unhealthy` e `fit`.
Stessi fold, stesso seed della v1 → il delta è attribuibile solo alla feature.

**Criterio di adozione**: delta OOF (dopo prior-correction) positivo e coerente sui fold.
Altrimenti si scarta, senza rimpianti.

**VERDETTO (eseguito)**: delta = 0.00000 spaccato. LightGBM sfrutta gia' il NaN di `bmi` negli split nativi; l'indicatore esplicito e' informazione ridondante. Feature scartata. Celle ricommentate per non sprecare compute nelle run successive.

In [ ]:
# X_v2      = make_X(train, add_bmi_missing=True)
# X_test_v2 = make_X(test,  add_bmi_missing=True)

# oof_v2, models_v2, _ = run_cv(X_v2, y, label='v2')
# t_star_v2, sweep_v2 = prior_sweep(oof_v2, y, priors)

# confronto onesto: delta a parita' di fold e seed, dopo prior-correction
# ba_v1 = max(sweep_v1.values())
# ba_v2 = max(sweep_v2.values())
# print(f'\nv1: {ba_v1:.5f} | v2: {ba_v2:.5f} | delta: {ba_v2 - ba_v1:+.5f}')

## 8. v3 — Optuna

Regole d'ingaggio:
1. **La funzione obiettivo ottimizza la BA *dopo* prior-correction** (sweep su t interno, costa solo argmax) — ottimizzare l'argmax nudo significherebbe ottimizzare la metrica sbagliata.
2. **Trial baseline in coda** (`enqueue_trial`) = paracadute anti-regressione: il best non potrà mai essere peggio della v1.
3. **Tuning a 3 fold** (vs 5 finali) + `MedianPruner` sui fold: con 690k righe il budget va speso con giudizio. I parametri vincenti vengono poi ri-validati con la pipeline standard a 5 fold, stessi fold della v1 → confronto pulito.

⏱️ ~30 trial × 3 fold su 690k righe: mettere in conto **qualche ora** su CPU Kaggle. Manopole: `N_TRIALS`, oppure `timeout=` in `study.optimize`.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_FOLDS = 3
N_TRIALS   = 30
T_GRID     = np.arange(0.85, 1.16, 0.05)

def ba_prior_corrected(y_true, proba, priors, ts=T_GRID):
    return max(balanced_accuracy_score(y_true, (proba / priors**t).argmax(1)) for t in ts)

def objective(trial):
    params = dict(
        objective='multiclass', num_class=3,
        n_estimators=3000, random_state=SEED, verbosity=-1, bagging_freq=1,
        learning_rate     = trial.suggest_float('learning_rate', 0.02, 0.10, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 31, 255, log=True),
        min_child_samples = trial.suggest_int('min_child_samples', 20, 300, log=True),
        feature_fraction  = trial.suggest_float('feature_fraction', 0.60, 1.00),
        bagging_fraction  = trial.suggest_float('bagging_fraction', 0.60, 1.00),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    )
    skf = StratifiedKFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X), 3))
    seen = np.zeros(len(X), dtype=bool)
    for i, (tr, va) in enumerate(skf.split(X, y)):
        m = lgb.LGBMClassifier(**params)
        m.fit(X.iloc[tr], y[tr],
              eval_set=[(X.iloc[va], y[va])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = m.predict_proba(X.iloc[va])
        seen[va] = True
        # BA parziale (sui fold visti finora) per il pruning
        trial.report(ba_prior_corrected(y[seen], oof[seen], priors), i)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return ba_prior_corrected(y, oof, priors)

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
)

# paracadute anti-regressione: la baseline v1 (con i default espliciti)
study.enqueue_trial(dict(
    learning_rate=0.05, num_leaves=63, min_child_samples=20,
    feature_fraction=1.0, bagging_fraction=1.0, reg_alpha=1e-3, reg_lambda=1e-3,
))

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'best trial #{study.best_trial.number}: BA (3-fold, post-corr) = {study.best_value:.5f}')
study.best_params

### Ri-validazione a 5 fold (stessi fold della v1) e confronto finale

Il punteggio a 3 fold e quello a 5 fold **non sono confrontabili tra loro**: il verdetto ufficiale
è sempre e solo la pipeline standard. Delta calcolato dopo prior-correction, a parità di fold.

In [ ]:
best_params = {**LGB_PARAMS, **study.best_params, 'n_estimators': 3000}

oof_v3, models_v3, _ = run_cv(X, y, params=best_params, label='v3')
t_star_v3, sweep_v3 = prior_sweep(oof_v3, y, priors, ts=np.arange(0.85, 1.16, 0.05))

ba_v1 = max(sweep_v1.values())
ba_v3 = max(sweep_v3.values())
print(f'\nv1: {ba_v1:.5f} | v3: {ba_v3:.5f} | delta: {ba_v3 - ba_v1:+.5f}')

### Submission v3

Solo se il delta è positivo e sensato; altrimenti la v1 resta la submission di riferimento
(e sarà il segnale che il margine va cercato altrove: ensemble con CatBoost).

In [ ]:
test_proba_v3 = np.mean([m.predict_proba(X_test) for m in models_v3], axis=0)
test_pred_v3  = le.classes_[(test_proba_v3 / priors**t_star_v3).argmax(1)]

sub = pd.DataFrame({'id': test['id'], TARGET: test_pred_v3})
sub.to_csv('submission.csv', index=False)
print(f't usato: {t_star_v3:.2f}')
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()

## 9. v4 — CatBoost (GPU)

Perché CatBoost e non "ancora tuning": il margine residuo è nella **diversità**, non nell'ottimizzazione.
CatBoost gestisce le categoriche con **ordered target statistics** (encoding basato sul target, calcolato
su permutazioni per evitare leakage) — un meccanismo genuinamente diverso dagli split su categoria di
LightGBM. Errori decorrelati = carburante per il blend.

Due differenze operative da non dimenticare:
1. **CatBoost NON accetta NaN nelle categoriche** (ironico, per un gatto): vanno riempiti con una
   categoria esplicita `'missing'`. I NaN numerici invece li gestisce nativamente, come LightGBM.
2. `task_type='GPU'`: supporto maturo, 690k righe diventano una passeggiata. È qui che si spendono
   le ore GPU risparmiate sul tuning LightGBM.

Pipeline identica alla v1/v3: stessi 5 fold, stesso seed, prior-correction in coda.

In [ ]:
from catboost import CatBoostClassifier

def make_X_cb(df):
    X_ = df.drop(columns=['id', TARGET], errors='ignore').copy()
    for c in cat_cols:
        X_[c] = X_[c].astype(object).fillna('missing').astype(str)
    return X_

X_cb      = make_X_cb(train)
X_test_cb = make_X_cb(test)

CB_PARAMS = dict(
    loss_function='MultiClass',
    iterations=3000, learning_rate=0.05, depth=8,
    task_type='GPU', devices='0',
    random_seed=SEED, verbose=0,
)

def run_cv_cb(X_, y, params=CB_PARAMS, n_folds=N_FOLDS, seed=SEED, label='cb'):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros((len(X_), 3))
    models, fold_scores = [], []
    for fold, (tr, va) in enumerate(skf.split(X_, y)):
        model = CatBoostClassifier(**params, cat_features=cat_cols)
        model.fit(X_.iloc[tr], y[tr],
                  eval_set=(X_.iloc[va], y[va]),
                  early_stopping_rounds=100, use_best_model=True)
        oof[va] = model.predict_proba(X_.iloc[va])
        models.append(model)
        ba = balanced_accuracy_score(y[va], oof[va].argmax(1))
        fold_scores.append(ba)
        print(f'[{label}] fold {fold}: BA = {ba:.5f}')
    print(f'[{label}] OOF BA (argmax): {balanced_accuracy_score(y, oof.argmax(1)):.5f}')
    return oof, models, fold_scores

oof_v4, models_v4, _ = run_cv_cb(X_cb, y, label='v4')
t_star_v4, sweep_v4 = prior_sweep(oof_v4, y, priors, ts=np.arange(0.85, 1.16, 0.05))

ba_v3 = max(sweep_v3.values())
ba_v4 = max(sweep_v4.values())
print(f'\nv3 (lgb): {ba_v3:.5f} | v4 (cb): {ba_v4:.5f} | delta: {ba_v4 - ba_v3:+.5f}')

### Diagnostica di diversità

Prima ancora del punteggio solo-CatBoost, la domanda che decide il destino del blend:
**quanto sono decorrelati gli errori dei due modelli?**
Se sbagliano sulle stesse righe, il blend non ha nulla da guadagnare — anche con un ottimo punteggio v4.
Un CatBoost leggermente peggiore ma *diverso* vale più di un clone.

In [ ]:
pred_lgb = (oof_v3 / priors**1.0).argmax(1)
pred_cb  = (oof_v4 / priors**t_star_v4).argmax(1)

err_lgb = pred_lgb != y
err_cb  = pred_cb  != y
both    = (err_lgb & err_cb).sum()
only_l  = (err_lgb & ~err_cb).sum()
only_c  = (~err_lgb & err_cb).sum()

print(f'errori lgb: {err_lgb.sum()} | errori cb: {err_cb.sum()}')
print(f'in comune: {both} | solo lgb: {only_l} | solo cb: {only_c}')
print(f'accordo predizioni: {(pred_lgb == pred_cb).mean():.4f}')
print(f'correlazione proba (media per classe): '
      f'{np.mean([np.corrcoef(oof_v3[:,k], oof_v4[:,k])[0,1] for k in range(3)]):.4f}')

## 10. v5 — Blend

Media pesata delle probabilità OOF: `w * lgb + (1-w) * cb`, con sweep congiunto di **w** e **t**
sull'OOF. Due soli iperparametri stimati su 690k righe di OOF: rischio di overfit trascurabile.

**Criterio**: il blend si adotta solo se batte il migliore dei due singoli.

In [ ]:
best = (-1, None, None)
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_v3 + (1 - w) * oof_v4
    for t in np.arange(0.85, 1.16, 0.05):
        ba = balanced_accuracy_score(y, (blend / priors**t).argmax(1))
        if ba > best[0]:
            best = (ba, round(float(w), 2), round(float(t), 2))

ba_v5, w_star, t_star_v5 = best
print(f'v5 blend: BA = {ba_v5:.5f} con w(lgb) = {w_star}, t = {t_star_v5}')
print(f'vs migliore singolo: {max(ba_v3, ba_v4):.5f} | delta: {ba_v5 - max(ba_v3, ba_v4):+.5f}')

### Submission v5 (o fallback)

Se il blend vince: media pesata delle probabilità di test dei due set di modelli, poi correzione con
(w*, t*). Se non vince, la submission resta quella del migliore singolo — nessun affezionamento.

In [ ]:
test_proba_lgb = np.mean([m.predict_proba(X_test)    for m in models_v3], axis=0)
test_proba_cb  = np.mean([m.predict_proba(X_test_cb) for m in models_v4], axis=0)

test_blend = w_star * test_proba_lgb + (1 - w_star) * test_proba_cb
test_pred  = le.classes_[(test_blend / priors**t_star_v5).argmax(1)]

sub = pd.DataFrame({'id': test['id'], TARGET: test_pred})
sub.to_csv('submission.csv', index=False)
print(f'blend: w(lgb)={w_star}, t={t_star_v5}')
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()